# Fraudulent Transaction Detection: XGBoost
CPE 232 Data Models (KMUTT), Kaggle competition, solo. Metric: **F1 Macro**.

Binary classification of transactions (`is_fraud`) on a simulated financial-transaction dataset (~5.4M training rows, ~0.12% fraud).

The competition data is restricted to enrolled students and is **not** included in this repo. To run on your own copy, set `DATA_DIR` to the folder containing `train.csv` and `test.csv`. To try the pipeline without the real data, use the synthetic files in `data/sample/` (`DATA_DIR=data/sample`).

The outputs below (missing-value counts, hold-out report) come from the original Kaggle run.

In [ ]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Folder containing train.csv and test.csv. Defaults to the Kaggle input path.
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input/competitions/fraud-detection-cpe-232-data-models")

## 1. Load data

In [ ]:
data = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
data_test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print(data.shape, data_test.shape)

## 2. Missing values
`transac_type`, `amount`, `src_bal` and `dst_bal` contain nulls (roughly 5-13% of rows).

In [4]:
data.isnull().sum()

id                       0
time_ind                 0
transac_type        648331
amount              702632
src_acc                  0
src_bal             269799
src_new_bal              0
dst_acc                  0
dst_bal             270191
dst_new_bal              0
is_fraud                 0
is_flagged_fraud         0
dtype: int64

## 3. Cleaning and feature engineering
- `transac_type` mapped to integers; missing values become an extra `Unknown` category.
- Account IDs factorized to integers; a flag marks merchant destinations (`M...` accounts).
- Missing numeric values filled with the column median.
- Balance-consistency errors (`error_src`, `error_dst`), amount-to-balance ratio, day/hour from `time_ind`, and account frequency counts.

In [ ]:
def cleaning(selected_data):
    df = selected_data.copy()
    
    # 1. Map 'transac_type' to numbers (keeps it in the same column)
    type_map = {'CASH_OUT': 0, 'TRANSFER': 1, 'PAYMENT': 2, 'CASH_IN': 3, 'DEBIT': 4, 'Unknown': 5}
    df['transac_type'] = df['transac_type'].astype(str).map(type_map).fillna(5).astype(int)
    
    # 2. Convert Account IDs to numbers
    # We use factorize here to turn 'C123...' into an integer ID
    df['src_acc'] = pd.factorize(df['src_acc'])[0]
    df['dst_acc'] = pd.factorize(df['dst_acc'])[0]

    # 3. Create the numeric Merchant flag
    df['is_merchant_dst'] = df['dst_acc_str'] = selected_data['dst_acc'].astype(str).str.startswith('M').astype(int)
    
    # 4. Handle Numeric Nulls
    for col in ['amount', 'src_bal', 'dst_bal']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

    # 5. Feature Engineering
    df['error_src'] = df['src_bal'] - df['amount'] - df['src_new_bal']
    df['error_dst'] = df['dst_bal'] + df['amount'] - df['dst_new_bal']
    df['amt_to_bal_ratio'] = np.where(df['src_bal'] > 0, df['amount'] / df['src_bal'], 0)
    df['day'] = (df['time_ind'] // 24) % 31  # Days (0-30)
    df['hour'] = df['time_ind'] % 24         # Hour of the day (0-23)
    src_counts = df['src_acc'].value_counts()
    dst_counts = df['dst_acc'].value_counts()
    
    df['src_freq'] = df['src_acc'].map(src_counts)
    df['dst_freq'] = df['dst_acc'].map(dst_counts)
    
    return df

In [ ]:
df = cleaning(data)

In [ ]:
df.shape

## 4. Train and evaluate (80/20 hold-out)
Default `XGBClassifier`; accuracy is uninformative at this class imbalance, so look at the fraud-class precision / recall / F1.

In [ ]:
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Initialize the XGBClassifier
model = xgb.XGBClassifier()
# Train the model
model.fit(X_train, y_train)
# Make predictions
y_pred = model.predict(X_test)

In [9]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.00
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1080327
           1       0.96      0.95      0.95      1319

    accuracy                           1.00   1081646
   macro avg       0.98      0.97      0.98   1081646
weighted avg       1.00      1.00      1.00   1081646

Confusion Matrix:
[[1080281      46]
 [     72    1247]]


## 5. Predict on the test set and write the submission

In [ ]:
df_test = cleaning(data_test)

In [ ]:
predictions = model.predict(df_test)

answer = pd.DataFrame()
answer["id"] = data_test["id"] 
answer["is_fraud"] = predictions

In [12]:
answer["is_fraud"].value_counts()


is_fraud
0    953201
1      1192
Name: count, dtype: int64

In [ ]:
answer.to_csv('answer.csv', index=False)